In [1]:
# ================================================================
# CELL 1 — Imports & Configuration
# ================================================================
# PRD §3.3 | Phase 1
#
# Threshold Stability Analysis in Deepfake Audio Detection
# Hypothesis: models exhibit stable AUC but unstable optimal thresholds
# ================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler
import numpy as np
import matplotlib
matplotlib.use('Agg')  # remove for interactive use
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy.interpolate import interp1d
from scipy.special import expit as sigmoid_fn
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, roc_curve, auc,
    brier_score_loss, confusion_matrix
)

# ── Hyperparameters ──────────────────────────────────────────────
OUTER_SPLITS = 5      # PRD §3.3: outer folds
INNER_SPLITS = 3      # PRD §3.3: inner folds
EPOCHS       = 5
PATIENCE     = 5
BATCH_SIZE   = 32
LR           = 5e-4
RANDOM_STATE =22
PLOT_DIR     = "."

# PRD §4 Experiment 4: perturbation settings
NOISE_LEVELS = [0.0, 0.01, 0.05, 0.10]   # Gaussian std added to features

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device       : {device}")
print(f"Outer folds  : {OUTER_SPLITS}  |  Inner folds: {INNER_SPLITS}")
print(f"Total inner training runs: {OUTER_SPLITS * INNER_SPLITS}")
print(f"Perturbation noise levels: {NOISE_LEVELS}")

Device       : cpu
Outer folds  : 5  |  Inner folds: 3
Total inner training runs: 15
Perturbation noise levels: [0.0, 0.01, 0.05, 0.1]


In [2]:
# ================================================================
# CELL 2 — Load Data
# ================================================================
# PRD §3.1: X (18105, 128, 94, 1) | Real: 7172 | Fake: 10933
# ================================================================
X = np.load("X_features.npy")   # (18105, 128, 94, 1)
y = np.load("y_labels.npy")     # (18105,)

print(f"Dataset  →  X: {X.shape}  |  y: {y.shape}")
print(f"Class 0 (real): {(y==0).sum()}  |  Class 1 (fake): {(y==1).sum()}")
print(f"Imbalance ratio: {(y==1).sum()/(y==0).sum():.3f}  (fake/real)")

Dataset  →  X: (18105, 128, 94, 1)  |  y: (18105,)
Class 0 (real): 7172  |  Class 1 (fake): 10933
Imbalance ratio: 1.524  (fake/real)


In [3]:
# ================================================================
# CELL 3 — Dataset & Model
# ================================================================
# PRD §3.2: Fixed lightweight CNN across ALL experiments.
# Architecture is not the contribution — threshold behaviour is.
# ================================================================

class AudioDataset(Dataset):
    """(N,H,W,C) numpy → (N,C,H,W) PyTorch tensors."""
    def __init__(self, features, labels):
        self.X = torch.from_numpy(features).permute(0, 3, 1, 2).float()
        self.y = torch.from_numpy(labels).float().unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


class DeepfakeDetector(nn.Module):
    """Fixed baseline CNN — 2 conv blocks + global pool + dense head."""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)
        self.pool  = nn.MaxPool2d(2, 2)
        self.ap    = nn.AdaptiveAvgPool2d((8, 8))
        self.fc    = nn.Sequential(
            nn.Linear(64*8*8, 128), nn.ReLU(), nn.Dropout(0.4), nn.Linear(128, 1)
        )
    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        return self.fc(torch.flatten(self.ap(x), 1))

print("Model and Dataset defined (fixed architecture per PRD §3.2).")

Model and Dataset defined (fixed architecture per PRD §3.2).


In [4]:
# ================================================================
# CELL 4 — Threshold & Metric Helper Functions
# ================================================================
# PRD §5: Core metrics + Spoofing metrics (EER, FAR/FRR)
#         + Stability metrics (threshold std, J-score variance)
# ================================================================

def get_probs(model, loader):
    """Inference → (probs, labels) arrays."""
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for bx, by in loader:
            p = torch.sigmoid(model(bx.to(device))).cpu().numpy().ravel()
            probs.extend(p)
            labels.extend(by.numpy().ravel())
    return np.array(probs), np.array(labels)


def youden_threshold(probs, labels):
    """Threshold maximising Youden J = TPR − FPR."""
    fpr, tpr, thresholds = roc_curve(labels, probs)
    j      = tpr - fpr
    idx    = np.argmax(j)
    return float(thresholds[idx]), float(j[idx]), fpr, tpr, thresholds


def eer_threshold(probs, labels):
    """
    Equal Error Rate threshold: the point where FAR == FRR.
    FAR = FPR = FP/(FP+TN)
    FRR = FNR = FN/(FN+TP) = 1 - TPR
    """
    fpr, tpr, thresholds = roc_curve(labels, probs)
    fnr = 1.0 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    eer_val   = float((fpr[idx] + fnr[idx]) / 2.0)
    eer_thresh = float(thresholds[idx])
    return eer_thresh, eer_val, fpr, tpr, fnr


def far_frr(probs, labels, threshold):
    """False Acceptance Rate and False Rejection Rate at a threshold."""
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    far = fp / (fp + tn + 1e-8)   # FAR = FPR
    frr = fn / (fn + tp + 1e-8)   # FRR = FNR
    return far, frr


def full_metrics(probs, labels, threshold, label=""):
    """All PRD §5 metrics at a given threshold."""
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    far, frr = far_frr(probs, labels, threshold)
    return {
        "label"      : label,
        "threshold"  : threshold,
        "accuracy"   : accuracy_score(labels, preds),
        "f1"         : f1_score(labels, preds, zero_division=0),
        "auc"        : roc_auc_score(labels, probs),
        "sensitivity": tp / (tp + fn + 1e-8),
        "specificity": tn / (tn + fp + 1e-8),
        "precision"  : tp / (tp + fp + 1e-8),
        "far"        : far,
        "frr"        : frr,
        "brier"      : brier_score_loss(labels, probs),
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
        "report"     : classification_report(labels, preds, digits=4),
    }


def platt_calibrate(train_logits, train_labels, test_logits):
    """
    Platt scaling: fit a logistic regression on logits → calibrated probs.
    Returns calibrated probabilities for test_logits.
    """
    from sklearn.linear_model import LogisticRegression
    lr = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)
    lr.fit(train_logits.reshape(-1, 1), train_labels)
    return lr.predict_proba(test_logits.reshape(-1, 1))[:, 1]


def temperature_scale(logits, temperature):
    """Divide logits by temperature before sigmoid → sharpens/softens probs."""
    return sigmoid_fn(logits / temperature)


def find_temperature(val_logits, val_labels, temps=None):
    """Grid-search temperature that minimises Brier score on val set."""
    if temps is None:
        temps = np.linspace(0.1, 5.0, 100)
    best_t, best_b = 1.0, float('inf')
    for t in temps:
        probs = temperature_scale(val_logits, t)
        b = brier_score_loss(val_labels, probs)
        if b < best_b:
            best_b, best_t = b, t
    return best_t


def ece_score(probs, labels, n_bins=10):
    """Expected Calibration Error (PRD §5 optional calibration metric)."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i+1])
        if mask.sum() == 0:
            continue
        acc  = labels[mask].mean()
        conf = probs[mask].mean()
        ece += mask.sum() * abs(acc - conf)
    return float(ece / len(probs))


print("All helper functions defined (Youden, EER, FAR/FRR, Platt, Temperature, ECE).")

All helper functions defined (Youden, EER, FAR/FRR, Platt, Temperature, ECE).


In [5]:
# ================================================================
# CELL 5 — Training Utility
# ================================================================

def train_fold(train_idx, val_idx, dataset):
    """
    Train a fresh model. Returns:
      best_state   : best model weights (by val AUC)
      val_logits   : raw logits on val set (needed for calibration)
      val_probs    : sigmoid(logits)
      val_labels   : ground truth
      history      : loss / acc / auc per epoch
      epochs_ran   : actual epochs completed
    """
    train_loader = DataLoader(dataset, batch_size=BATCH_SIZE,
                              sampler=SubsetRandomSampler(train_idx))
    val_loader   = DataLoader(dataset, batch_size=BATCH_SIZE,
                              sampler=SubsetRandomSampler(val_idx))

    model     = DeepfakeDetector().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.BCEWithLogitsLoss()

    best_auc, best_state = -1, None
    patience_ctr = 0
    history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_auc": []}

    for epoch in range(1, EPOCHS + 1):
        model.train()
        t_loss = 0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward(); optimizer.step()
            t_loss += loss.item()
        history["train_loss"].append(t_loss / len(train_loader))

        model.eval()
        v_loss = 0
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(device), by.to(device)
                v_loss += criterion(model(bx), by).item()
        history["val_loss"].append(v_loss / len(val_loader))

        vp, vl = get_probs(model, val_loader)
        thresh, _, _, _, _ = youden_threshold(vp, vl)
        v_acc  = accuracy_score(vl, (vp >= thresh).astype(int))
        v_auc  = roc_auc_score(vl, vp)
        history["val_acc"].append(v_acc)
        history["val_auc"].append(v_auc)

        if v_auc > best_auc:
            best_auc   = v_auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                break

    # ── Get final logits from best checkpoint ──────────────────────
    model.load_state_dict(best_state)
    model.eval()
    logits_list, probs_list, labels_list = [], [], []
    with torch.no_grad():
        for bx, by in val_loader:
            raw = model(bx.to(device)).cpu().numpy().ravel()
            logits_list.extend(raw)
            probs_list.extend(sigmoid_fn(raw))
            labels_list.extend(by.numpy().ravel())

    return (best_state,
            np.array(logits_list), np.array(probs_list), np.array(labels_list),
            history, epoch)

print("train_fold() defined.")

train_fold() defined.


In [6]:
# ================================================================
# CELL 6 — Nested K-Fold: Core Evaluation Loop
# ================================================================
# PRD §3.3 Evaluation Pipeline
# Inner loop  → train + compute Youden & EER on val
# Outer loop  → apply thresholds to held-out test, record all metrics
# PRD §4 Exp 1, 2, 3 data collected here
# ================================================================

full_dataset = AudioDataset(X, y)
outer_skf    = StratifiedKFold(n_splits=OUTER_SPLITS, shuffle=True,
                                random_state=RANDOM_STATE)
inner_skf    = StratifiedKFold(n_splits=INNER_SPLITS, shuffle=True,
                                random_state=RANDOM_STATE)

# ── Storage ──────────────────────────────────────────────────────
fold_records   = []   # one dict per outer fold — all metrics for all methods
roc_curves     = []   # (fpr, tpr, auc) per outer fold
det_curves     = []   # (fpr, fnr) per outer fold
agg_cm_youden  = np.zeros((2,2), dtype=int)
agg_cm_eer     = np.zeros((2,2), dtype=int)

best_global_auc   = -1
best_model_state  = None
best_outer_fold   = None

print("=" * 72)
print(f"  NESTED K-FOLD THRESHOLD STABILITY ANALYSIS")
print(f"  Outer: {OUTER_SPLITS}  Inner: {INNER_SPLITS}  "
      f"Total inner runs: {OUTER_SPLITS*INNER_SPLITS}")
print("=" * 72)

for outer_idx, (trainval_idx, test_idx) in enumerate(
        outer_skf.split(X, y), start=1):

    print(f"\n{'━'*72}")
    print(f"  OUTER FOLD {outer_idx}/{OUTER_SPLITS}  "
          f"trainval={len(trainval_idx)}  test={len(test_idx)}")
    print(f"{'━'*72}")

    X_tv = X[trainval_idx]
    y_tv = y[trainval_idx]

    # ── Inner loop ───────────────────────────────────────────────
    inner_best_auc   = -1
    inner_best_state = None
    # Collect inner val thresholds (Youden + EER) for stability tracking
    inner_youden_thresholds = []
    inner_eer_thresholds    = []
    inner_j_scores          = []
    inner_eer_vals          = []

    for inner_idx, (tr_rel, val_rel) in enumerate(
            inner_skf.split(X_tv, y_tv), start=1):

        tr_idx  = trainval_idx[tr_rel]
        val_idx = trainval_idx[val_rel]

        print(f"  ├─ Inner {inner_idx}/{INNER_SPLITS}  "
              f"train={len(tr_idx)}  val={len(val_idx)}")

        state, vlogits, vprobs, vlabels, hist, ep = train_fold(
            tr_idx, val_idx, full_dataset)

        # Thresholds on inner val
        y_thresh, j_val, _, _, _ = youden_threshold(vprobs, vlabels)
        e_thresh, eer_val, _, _, _ = eer_threshold(vprobs, vlabels)
        v_auc = roc_auc_score(vlabels, vprobs)

        inner_youden_thresholds.append(y_thresh)
        inner_eer_thresholds.append(e_thresh)
        inner_j_scores.append(j_val)
        inner_eer_vals.append(eer_val)

        print(f"  │   Ep={ep:2d}  AUC={v_auc:.4f}  "
              f"Youden={y_thresh:.4f} (J={j_val:.4f})  "
              f"EER={e_thresh:.4f} (EER={eer_val:.4f})")

        if v_auc > inner_best_auc:
            inner_best_auc   = v_auc
            inner_best_state = state
            # Store logits/labels for calibration fitting on inner val
            best_inner_logits = vlogits
            best_inner_labels = vlabels
            best_inner_probs  = vprobs

    # ── Threshold strategy: use mean of inner thresholds ──────────
    final_youden_thresh = np.mean(inner_youden_thresholds)
    final_eer_thresh    = np.mean(inner_eer_thresholds)

    # ── Apply to outer TEST set ───────────────────────────────────
    model = DeepfakeDetector().to(device)
    model.load_state_dict(inner_best_state)
    model.eval()

    test_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE,
                             sampler=SubsetRandomSampler(test_idx))

    test_logits, test_probs, test_labels = [], [], []
    with torch.no_grad():
        for bx, by in test_loader:
            raw = model(bx.to(device)).cpu().numpy().ravel()
            test_logits.extend(raw)
            test_probs.extend(sigmoid_fn(raw))
            test_labels.extend(by.numpy().ravel())

    test_logits = np.array(test_logits)
    test_probs  = np.array(test_probs)
    test_labels = np.array(test_labels)

    # ── Calibration (Platt + Temperature) ────────────────────────
    platt_test_probs = platt_calibrate(
        best_inner_logits, best_inner_labels, test_logits)
    best_temp = find_temperature(best_inner_logits, best_inner_labels)
    temp_test_probs = temperature_scale(test_logits, best_temp)

    # ── Youden & EER on calibrated probs ─────────────────────────
    platt_thresh, _, _, _, _ = youden_threshold(platt_test_probs, test_labels)
    temp_thresh,  _, _, _, _ = youden_threshold(temp_test_probs,  test_labels)
    eer_on_test, eer_val_test, _, _, _ = eer_threshold(test_probs, test_labels)

    # ── Metrics for all strategies ────────────────────────────────
    m_youden = full_metrics(test_probs,       test_labels, final_youden_thresh, "Youden")
    m_eer    = full_metrics(test_probs,       test_labels, final_eer_thresh,    "EER")
    m_fixed  = full_metrics(test_probs,       test_labels, 0.5,                 "Fixed-0.5")
    m_platt  = full_metrics(platt_test_probs, test_labels, platt_thresh,        "Platt+Youden")
    m_temp   = full_metrics(temp_test_probs,  test_labels, temp_thresh,         "Temp+Youden")

    # ── ROC + DET curve data ──────────────────────────────────────
    fpr_r, tpr_r, _ = roc_curve(test_labels, test_probs)
    fnr_r = 1.0 - tpr_r
    roc_curves.append((fpr_r, tpr_r, auc(fpr_r, tpr_r)))
    det_curves.append((fpr_r, fnr_r))

    agg_cm_youden += confusion_matrix(
        test_labels, (test_probs >= final_youden_thresh).astype(int))
    agg_cm_eer    += confusion_matrix(
        test_labels, (test_probs >= final_eer_thresh).astype(int))

    rec = {
        "outer_fold"              : outer_idx,
        # Inner threshold stability data (PRD §4 Exp 1)
        "inner_youden_thresholds" : inner_youden_thresholds,
        "inner_eer_thresholds"    : inner_eer_thresholds,
        "inner_j_scores"          : inner_j_scores,
        "inner_eer_vals"          : inner_eer_vals,
        "youden_thresh_mean"      : final_youden_thresh,
        "youden_thresh_std"       : np.std(inner_youden_thresholds),
        "eer_thresh_mean"         : final_eer_thresh,
        "eer_thresh_std"          : np.std(inner_eer_thresholds),
        "j_score_std"             : np.std(inner_j_scores),
        # Calibration params
        "best_temperature"        : best_temp,
        "ece_raw"                 : ece_score(test_probs,       test_labels),
        "ece_platt"               : ece_score(platt_test_probs, test_labels),
        "ece_temp"                : ece_score(temp_test_probs,  test_labels),
        # Per-strategy metrics
        "youden" : m_youden,
        "eer"    : m_eer,
        "fixed"  : m_fixed,
        "platt"  : m_platt,
        "temp"   : m_temp,
    }
    fold_records.append(rec)

    if m_youden["auc"] > best_global_auc:
        best_global_auc  = m_youden["auc"]
        best_model_state = inner_best_state
        best_outer_fold  = outer_idx

    print(f"\n  ✦ Outer {outer_idx} TEST RESULTS")
    print(f"    {'Method':<15} {'Thresh':>8} {'Acc':>7} "
          f"{'F1':>7} {'AUC':>7} {'Sens':>7} {'Spec':>7} "
          f"{'FAR':>7} {'FRR':>7}")
    for m in [m_youden, m_eer, m_fixed, m_platt, m_temp]:
        print(f"    {m['label']:<15} {m['threshold']:>8.4f} "
              f"{m['accuracy']:>7.4f} {m['f1']:>7.4f} "
              f"{m['auc']:>7.4f} {m['sensitivity']:>7.4f} "
              f"{m['specificity']:>7.4f} "
              f"{m['far']:>7.4f} {m['frr']:>7.4f}")
    print(f"    Inner Youden thresh std : {rec['youden_thresh_std']:.4f}")
    print(f"    Inner EER   thresh std  : {rec['eer_thresh_std']:.4f}")
    print(f"    Temperature             : {best_temp:.3f}")
    print(f"    ECE raw/Platt/Temp      : "
          f"{rec['ece_raw']:.4f} / {rec['ece_platt']:.4f} / {rec['ece_temp']:.4f}")

print("\n" + "=" * 72)
print(f"  All {OUTER_SPLITS} outer folds complete.")
print("=" * 72)

  NESTED K-FOLD THRESHOLD STABILITY ANALYSIS
  Outer: 5  Inner: 3  Total inner runs: 15

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  OUTER FOLD 1/5  trainval=14484  test=3621
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ├─ Inner 1/3  train=9656  val=4828
  │   Ep= 5  AUC=0.9790  Youden=0.7073 (J=0.8489)  EER=0.7365 (EER=0.0776)
  ├─ Inner 2/3  train=9656  val=4828
  │   Ep= 5  AUC=0.9695  Youden=0.3982 (J=0.8000)  EER=0.3948 (EER=0.1004)
  ├─ Inner 3/3  train=9656  val=4828
  │   Ep= 5  AUC=0.9756  Youden=0.5579 (J=0.8350)  EER=0.6176 (EER=0.0839)

  ✦ Outer 1 TEST RESULTS
    Method            Thresh     Acc      F1     AUC    Sens    Spec     FAR     FRR
    Youden            0.5545  0.9213  0.9370  0.9790  0.9689  0.8487  0.1513  0.0311
    EER               0.5830  0.9229  0.9378  0.9790  0.9616  0.8640  0.1360  0.0384
    Fixed-0.5         0.5000  0.9144  0.9322  0.9790  0.9749  0.8222  0.1778  0.0251
    Platt+Youden   

In [7]:
# ================================================================
# CELL 7 — Experiment 1: Threshold Variability
# ================================================================
# PRD §4 Exp 1: Quantify how thresholds vary across folds
# Output: Table + boxplot
# ================================================================

print("\n" + "=" * 72)
print("  EXP 1 — THRESHOLD VARIABILITY ACROSS FOLDS")
print("=" * 72)
print("  (Threshold instability is the KEY CONTRIBUTION per PRD §9)\n")

# Collect all inner-fold thresholds across all outer folds
all_inner_youden = [t for r in fold_records for t in r["inner_youden_thresholds"]]
all_inner_eer    = [t for r in fold_records for t in r["inner_eer_thresholds"]]
all_outer_youden = [r["youden_thresh_mean"] for r in fold_records]
all_outer_eer    = [r["eer_thresh_mean"]    for r in fold_records]
all_j_scores     = [t for r in fold_records for t in r["inner_j_scores"]]
all_eer_vals     = [t for r in fold_records for t in r["inner_eer_vals"]]

print(f"  {'Metric':<30} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8} {'Range':>8}")
print(f"  {'-'*70}")
for name, vals in [
    ("Youden thresh (inner folds)",  all_inner_youden),
    ("EER    thresh (inner folds)",  all_inner_eer),
    ("Youden thresh (outer/test)",   all_outer_youden),
    ("EER    thresh (outer/test)",   all_outer_eer),
    ("J-score (inner folds)",        all_j_scores),
    ("EER value (inner folds)",      all_eer_vals),
]:
    v = np.array(vals)
    print(f"  {name:<30} {v.mean():>8.4f} {v.std():>8.4f} "
          f"{v.min():>8.4f} {v.max():>8.4f} {v.max()-v.min():>8.4f}")

print()
y_std = np.std(all_inner_youden)
e_std = np.std(all_inner_eer)
sig_threshold = 0.05  # conservative significance bar
print(f"  Youden std={y_std:.4f}  {'✅ SIGNIFICANT' if y_std > sig_threshold else '⚠ below 0.05'}")
print(f"  EER    std={e_std:.4f}  {'✅ SIGNIFICANT' if e_std > sig_threshold else '⚠ below 0.05'}")

# ── Boxplot ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: per-inner-fold thresholds grouped by outer fold
ax = axes[0]
youden_by_outer = [r["inner_youden_thresholds"] for r in fold_records]
eer_by_outer    = [r["inner_eer_thresholds"]    for r in fold_records]
positions_y = np.arange(1, OUTER_SPLITS+1) - 0.18
positions_e = np.arange(1, OUTER_SPLITS+1) + 0.18
bp1 = ax.boxplot(youden_by_outer, positions=positions_y, widths=0.3,
                 patch_artist=True,
                 boxprops=dict(facecolor='#5c8fe0', alpha=0.7),
                 medianprops=dict(color='black', lw=2))
bp2 = ax.boxplot(eer_by_outer, positions=positions_e, widths=0.3,
                 patch_artist=True,
                 boxprops=dict(facecolor='#e05c5c', alpha=0.7),
                 medianprops=dict(color='black', lw=2))
ax.set_xticks(range(1, OUTER_SPLITS+1))
ax.set_xticklabels([f"Outer {i}" for i in range(1, OUTER_SPLITS+1)])
ax.set_ylabel("Threshold Value", fontsize=11)
ax.set_title("Threshold Distribution per Outer Fold\n"
             "(each box = inner folds)", fontsize=11, fontweight='bold')
ax.legend([bp1['boxes'][0], bp2['boxes'][0]],
          ['Youden', 'EER'], fontsize=10)
ax.grid(True, linestyle='--', alpha=0.3, axis='y')

# Right: histogram of all thresholds
ax = axes[1]
ax.hist(all_inner_youden, bins=15, alpha=0.65, color='#5c8fe0',
        label=f"Youden  μ={np.mean(all_inner_youden):.3f}  σ={np.std(all_inner_youden):.3f}")
ax.hist(all_inner_eer, bins=15, alpha=0.65, color='#e05c5c',
        label=f"EER     μ={np.mean(all_inner_eer):.3f}  σ={np.std(all_inner_eer):.3f}")
ax.axvline(np.mean(all_inner_youden), color='#2a5aaa', lw=2, linestyle='--')
ax.axvline(np.mean(all_inner_eer),    color='#aa2a2a', lw=2, linestyle='--')
ax.axvline(0.5, color='grey', lw=1.5, linestyle=':', label="Fixed 0.5")
ax.set_xlabel("Threshold Value", fontsize=11)
ax.set_ylabel("Count", fontsize=11)
ax.set_title("Threshold Distribution — All Inner Folds", fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.3)

plt.suptitle("Exp 1: Threshold Variability (PRD §4)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/exp1_threshold_variability.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → exp1_threshold_variability.png")


  EXP 1 — THRESHOLD VARIABILITY ACROSS FOLDS
  (Threshold instability is the KEY CONTRIBUTION per PRD §9)

  Metric                             Mean      Std      Min      Max    Range
  ----------------------------------------------------------------------
  Youden thresh (inner folds)      0.4530   0.2277   0.0508   0.8406   0.7899
  EER    thresh (inner folds)      0.4911   0.2160   0.0881   0.8358   0.7477
  Youden thresh (outer/test)       0.4530   0.1220   0.2255   0.5591   0.3336
  EER    thresh (outer/test)       0.4911   0.1027   0.2988   0.5830   0.2841
  J-score (inner folds)            0.8194   0.0253   0.7697   0.8634   0.0938
  EER value (inner folds)          0.0930   0.0125   0.0704   0.1184   0.0480

  Youden std=0.2277  ✅ SIGNIFICANT
  EER    std=0.2160  ✅ SIGNIFICANT
Saved → exp1_threshold_variability.png


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_21080\1364213137.py:87: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# ================================================================
# CELL 8 — Experiment 2: AUC vs Threshold Stability
# ================================================================
# PRD §4 Exp 2: Show disconnect between ranking and decision stability
# "High AUC ≠ reliable deployment" — the core argument of the paper
# ================================================================

auc_per_fold    = [r["youden"]["auc"]        for r in fold_records]
ythresh_per_fold= [r["youden_thresh_mean"]   for r in fold_records]
ythresh_std_fold= [r["youden_thresh_std"]    for r in fold_records]
eer_per_fold    = [r["eer_thresh_mean"]      for r in fold_records]
eer_std_fold    = [r["eer_thresh_std"]       for r in fold_records]

print("\n" + "=" * 72)
print("  EXP 2 — AUC (stable) vs THRESHOLD (unstable)")
print("=" * 72)
print(f"  AUC          : mean={np.mean(auc_per_fold):.4f}  "
      f"std={np.std(auc_per_fold):.4f}  "
      f"range={max(auc_per_fold)-min(auc_per_fold):.4f}")
print(f"  Youden thresh: mean={np.mean(ythresh_per_fold):.4f}  "
      f"std={np.std(ythresh_per_fold):.4f}  "
      f"range={max(ythresh_per_fold)-min(ythresh_per_fold):.4f}")
print(f"  EER thresh   : mean={np.mean(eer_per_fold):.4f}  "
      f"std={np.std(eer_per_fold):.4f}  "
      f"range={max(eer_per_fold)-min(eer_per_fold):.4f}")
ratio = np.std(ythresh_per_fold) / (np.std(auc_per_fold) + 1e-8)
print(f"\n  Threshold std / AUC std ratio: {ratio:.2f}x")
print("  (ratio >> 1 confirms instability hypothesis)")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
folds = np.arange(1, OUTER_SPLITS+1)

# Left: AUC per fold
ax = axes[0]
ax.bar(folds, auc_per_fold, color='#2ca02c', alpha=0.75, edgecolor='black')
ax.axhline(np.mean(auc_per_fold), color='red', lw=2, linestyle='--',
           label=f"Mean={np.mean(auc_per_fold):.4f}")
ax.fill_between([0.5, OUTER_SPLITS+0.5],
                np.mean(auc_per_fold)-np.std(auc_per_fold),
                np.mean(auc_per_fold)+np.std(auc_per_fold),
                color='red', alpha=0.1, label=f"± std={np.std(auc_per_fold):.4f}")
ax.set_ylim(0, 1.05); ax.set_xticks(folds)
ax.set_xlabel("Outer Fold"); ax.set_ylabel("AUC")
ax.set_title("AUC per Fold\n(stable ✅)", fontsize=11, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, linestyle='--', alpha=0.3, axis='y')

# Middle: Youden threshold per fold with inner-fold std as error bars
ax = axes[1]
ax.bar(folds, ythresh_per_fold, yerr=ythresh_std_fold,
       color='#5c8fe0', alpha=0.75, edgecolor='black',
       capsize=6, error_kw={'elinewidth':2})
ax.axhline(np.mean(ythresh_per_fold), color='red', lw=2, linestyle='--',
           label=f"Mean={np.mean(ythresh_per_fold):.4f}")
ax.axhline(0.5, color='grey', lw=1.5, linestyle=':', label="Fixed 0.5")
ax.set_ylim(0, 1.05); ax.set_xticks(folds)
ax.set_xlabel("Outer Fold"); ax.set_ylabel("Youden Threshold")
ax.set_title("Youden Threshold per Fold\n(error bars = inner std  ⚠ unstable)",
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, linestyle='--', alpha=0.3, axis='y')

# Right: side-by-side std comparison (the key figure)
ax = axes[2]
categories = ['AUC', 'Youden\nThreshold', 'EER\nThreshold']
stds = [np.std(auc_per_fold), np.std(ythresh_per_fold), np.std(eer_per_fold)]
colors = ['#2ca02c', '#5c8fe0', '#e05c5c']
bars = ax.bar(categories, stds, color=colors, alpha=0.8, edgecolor='black')
for bar, val in zip(bars, stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f"{val:.4f}", ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.axhline(0.05, color='red', lw=1.5, linestyle='--', alpha=0.7,
           label="Significance bar (0.05)")
ax.set_ylabel("Standard Deviation", fontsize=11)
ax.set_title("Stability Comparison\nAUC std vs Threshold std",
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, linestyle='--', alpha=0.3, axis='y')

plt.suptitle("Exp 2: High AUC ≠ Reliable Deployment  (PRD §1.1)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/exp2_auc_vs_threshold_stability.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved → exp2_auc_vs_threshold_stability.png")


  EXP 2 — AUC (stable) vs THRESHOLD (unstable)
  AUC          : mean=0.9774  std=0.0041  range=0.0119
  Youden thresh: mean=0.4530  std=0.1220  range=0.3336
  EER thresh   : mean=0.4911  std=0.1027  range=0.2841

  Threshold std / AUC std ratio: 29.62x
  (ratio >> 1 confirms instability hypothesis)


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_21080\1608400840.py:79: UserWarning: Glyph 9989 (\N{WHITE HEAVY CHECK MARK}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_21080\1608400840.py:80: UserWarning: Glyph 9989 (\N{WHITE HEAVY CHECK MARK}) missing from font(s) DejaVu Sans.
  plt.savefig(f"{PLOT_DIR}/exp2_auc_vs_threshold_stability.png",


Saved → exp2_auc_vs_threshold_stability.png


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_21080\1608400840.py:82: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# ================================================================
# CELL 9 — Experiment 3: Threshold Strategy Comparison
# ================================================================
# PRD §4 Exp 3: Compare Youden vs EER vs Fixed-0.5 vs Calibrated
# PRD §5: F1, Accuracy, Sensitivity, Specificity, FAR, FRR
# ================================================================

strategies = ["youden", "eer", "fixed", "platt", "temp"]
strategy_labels = {
    "youden": "Youden's J",
    "eer"   : "EER",
    "fixed" : "Fixed 0.5",
    "platt" : "Platt+Youden",
    "temp"  : "Temp+Youden",
}
metric_keys = ["accuracy", "f1", "sensitivity", "specificity", "far", "frr", "threshold"]

print("\n" + "=" * 72)
print("  EXP 3 — THRESHOLD STRATEGY COMPARISON")
print("=" * 72)

rows = []
for strat in strategies:
    vals_per_metric = {k: [r[strat][k] for r in fold_records] for k in metric_keys}
    row = {"Strategy": strategy_labels[strat]}
    for k in metric_keys:
        v = np.array(vals_per_metric[k])
        row[k] = f"{v.mean():.4f} ± {v.std():.4f}"
    rows.append(row)

df_compare = pd.DataFrame(rows).set_index("Strategy")
print(df_compare.to_string())

# ── Grouped bar chart ─────────────────────────────────────────────
plot_metrics = ["accuracy", "f1", "sensitivity", "specificity"]
n_strats = len(strategies)
x = np.arange(len(plot_metrics))
width = 0.15
palette = ['#5c8fe0','#e05c5c','#9467bd','#2ca02c','#ff7f0e']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: performance metrics
ax = axes[0]
for i, (strat, color) in enumerate(zip(strategies, palette)):
    means = [np.mean([r[strat][m] for r in fold_records]) for m in plot_metrics]
    stds  = [np.std( [r[strat][m] for r in fold_records]) for m in plot_metrics]
    offset = (i - n_strats/2 + 0.5) * width
    ax.bar(x + offset, means, width=width, yerr=stds,
           label=strategy_labels[strat], color=color,
           alpha=0.8, capsize=4, error_kw={'elinewidth':1.2})

ax.set_xticks(x)
ax.set_xticklabels([m.capitalize() for m in plot_metrics], fontsize=10)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score (mean ± std across folds)", fontsize=10)
ax.set_title("Performance Metrics by Strategy", fontsize=11, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, linestyle='--', alpha=0.3, axis='y')

# Right: FAR vs FRR tradeoff
ax = axes[1]
for i, (strat, color) in enumerate(zip(strategies, palette)):
    far_vals = [r[strat]["far"] for r in fold_records]
    frr_vals = [r[strat]["frr"] for r in fold_records]
    ax.scatter(np.mean(far_vals), np.mean(frr_vals),
               s=200, color=color, zorder=5,
               label=f"{strategy_labels[strat]}  "
                     f"FAR={np.mean(far_vals):.3f}  FRR={np.mean(frr_vals):.3f}")
    ax.errorbar(np.mean(far_vals), np.mean(frr_vals),
                xerr=np.std(far_vals), yerr=np.std(frr_vals),
                color=color, alpha=0.5, capsize=4)

ax.plot([0,1],[0,1],'k--', lw=1, alpha=0.4, label="FAR = FRR (EER line)")
ax.set_xlabel("FAR (False Acceptance Rate)", fontsize=10)
ax.set_ylabel("FRR (False Rejection Rate)", fontsize=10)
ax.set_title("FAR vs FRR Tradeoff per Strategy", fontsize=11, fontweight='bold')
ax.legend(fontsize=7.5, loc='upper right')
ax.grid(True, linestyle='--', alpha=0.3)
ax.set_xlim(-0.01, 0.6); ax.set_ylim(-0.01, 0.6)

plt.suptitle("Exp 3: Threshold Strategy Comparison (PRD §4)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/exp3_strategy_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → exp3_strategy_comparison.png")


  EXP 3 — THRESHOLD STRATEGY COMPARISON
                     accuracy               f1      sensitivity      specificity              far              frr        threshold
Strategy                                                                                                                           
Youden's J    0.9130 ± 0.0115  0.9282 ± 0.0108  0.9344 ± 0.0349  0.8802 ± 0.0424  0.1198 ± 0.0424  0.0656 ± 0.0349  0.4530 ± 0.1220
EER           0.9132 ± 0.0147  0.9275 ± 0.0139  0.9238 ± 0.0370  0.8970 ± 0.0338  0.1030 ± 0.0338  0.0762 ± 0.0370  0.4911 ± 0.1027
Fixed 0.5     0.9100 ± 0.0136  0.9249 ± 0.0130  0.9207 ± 0.0422  0.8936 ± 0.0538  0.1064 ± 0.0538  0.0793 ± 0.0422  0.5000 ± 0.0000
Platt+Youden  0.9212 ± 0.0111  0.9343 ± 0.0097  0.9295 ± 0.0214  0.9085 ± 0.0226  0.0915 ± 0.0226  0.0705 ± 0.0214  0.5619 ± 0.0478
Temp+Youden   0.9212 ± 0.0111  0.9343 ± 0.0097  0.9295 ± 0.0214  0.9085 ± 0.0226  0.0915 ± 0.0226  0.0705 ± 0.0214  0.4878 ± 0.2172
Saved → exp3_strategy_comparison.pn

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_21080\1068583006.py:86: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# ================================================================
# CELL 10 — Experiment 4: Data Perturbation Sensitivity
# ================================================================
# PRD §4 Exp 4: Test robustness of thresholds under noise
# Perturbation: add Gaussian noise at multiple SNR levels
# Measure: threshold shift, performance degradation
# ================================================================

print("\n" + "=" * 72)
print("  EXP 4 — DATA PERTURBATION SENSITIVITY")
print(f"  Noise levels (Gaussian std): {NOISE_LEVELS}")
print("=" * 72)

# Use best model from nested CV
model_best = DeepfakeDetector().to(device)
model_best.load_state_dict(best_model_state)
model_best.eval()

# Reference: clean data
full_loader_clean = DataLoader(AudioDataset(X, y),
                               batch_size=BATCH_SIZE, shuffle=False)

perturb_results = []

for noise_std in NOISE_LEVELS:
    X_noisy = X + np.random.default_rng(42).normal(0, noise_std, X.shape)
    noisy_loader = DataLoader(AudioDataset(X_noisy.astype(np.float32), y),
                              batch_size=BATCH_SIZE, shuffle=False)

    nprobs, nlabels = get_probs(model_best, noisy_loader)
    y_thresh, j_val, _, _, _ = youden_threshold(nprobs, nlabels)
    e_thresh, eer_val, _, _, _ = eer_threshold(nprobs, nlabels)
    auc_val = roc_auc_score(nlabels, nprobs)
    m = full_metrics(nprobs, nlabels, y_thresh, f"noise={noise_std}")

    perturb_results.append({
        "noise_std"     : noise_std,
        "auc"           : auc_val,
        "youden_thresh" : y_thresh,
        "eer_thresh"    : e_thresh,
        "eer_val"       : eer_val,
        "j_val"         : j_val,
        "accuracy"      : m["accuracy"],
        "f1"            : m["f1"],
        "sensitivity"   : m["sensitivity"],
        "specificity"   : m["specificity"],
        "far"           : m["far"],
        "frr"           : m["frr"],
    })
    print(f"  noise={noise_std:.2f}  AUC={auc_val:.4f}  "
          f"Youden={y_thresh:.4f}  EER={e_thresh:.4f}  "
          f"F1={m['f1']:.4f}  FAR={m['far']:.4f}  FRR={m['frr']:.4f}")

clean = perturb_results[0]
worst = perturb_results[-1]
print(f"\n  Youden threshold shift (clean→max noise): "
      f"{clean['youden_thresh']:.4f} → {worst['youden_thresh']:.4f}  "
      f"(Δ={abs(worst['youden_thresh']-clean['youden_thresh']):.4f})")
print(f"  AUC degradation:  {clean['auc']:.4f} → {worst['auc']:.4f}  "
      f"(Δ={abs(worst['auc']-clean['auc']):.4f})")

# ── Plot ──────────────────────────────────────────────────────────
noises = [r["noise_std"]     for r in perturb_results]
aucs   = [r["auc"]           for r in perturb_results]
y_ts   = [r["youden_thresh"] for r in perturb_results]
e_ts   = [r["eer_thresh"]    for r in perturb_results]
f1s    = [r["f1"]            for r in perturb_results]
fars   = [r["far"]           for r in perturb_results]
frrs   = [r["frr"]           for r in perturb_results]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

ax = axes[0]
ax.plot(noises, aucs, 'o-', color='#2ca02c', lw=2, ms=8, label="AUC")
ax.plot(noises, f1s,  's-', color='#9467bd', lw=2, ms=8, label="F1")
ax.set_xlabel("Gaussian Noise Std"); ax.set_ylabel("Score")
ax.set_title("Performance Degradation\nunder Noise", fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.05); ax.legend(); ax.grid(True, linestyle='--', alpha=0.3)

ax = axes[1]
ax.plot(noises, y_ts, 'o-', color='#5c8fe0', lw=2, ms=8, label="Youden Threshold")
ax.plot(noises, e_ts, 's-', color='#e05c5c', lw=2, ms=8, label="EER Threshold")
ax.axhline(0.5, color='grey', lw=1.5, linestyle=':', label="Fixed 0.5")
ax.set_xlabel("Gaussian Noise Std"); ax.set_ylabel("Threshold Value")
ax.set_title("Threshold Shift under Noise", fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.05); ax.legend(); ax.grid(True, linestyle='--', alpha=0.3)

ax = axes[2]
ax.plot(noises, fars, 'o-', color='#e05c5c', lw=2, ms=8, label="FAR")
ax.plot(noises, frrs, 's-', color='#ff7f0e', lw=2, ms=8, label="FRR")
ax.set_xlabel("Gaussian Noise Std"); ax.set_ylabel("Rate")
ax.set_title("FAR / FRR under Noise", fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.05); ax.legend(); ax.grid(True, linestyle='--', alpha=0.3)

plt.suptitle("Exp 4: Data Perturbation Sensitivity (PRD §4)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/exp4_perturbation_sensitivity.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved → exp4_perturbation_sensitivity.png")


  EXP 4 — DATA PERTURBATION SENSITIVITY
  Noise levels (Gaussian std): [0.0, 0.01, 0.05, 0.1]
  noise=0.00  AUC=0.9837  Youden=0.4086  EER=0.4271  F1=0.9447  FAR=0.0712  FRR=0.0629
  noise=0.01  AUC=0.9836  Youden=0.4084  EER=0.4263  F1=0.9447  FAR=0.0710  FRR=0.0632
  noise=0.05  AUC=0.9836  Youden=0.4084  EER=0.4226  F1=0.9443  FAR=0.0700  FRR=0.0645
  noise=0.10  AUC=0.9835  Youden=0.3977  EER=0.4148  F1=0.9443  FAR=0.0703  FRR=0.0642

  Youden threshold shift (clean→max noise): 0.4086 → 0.3977  (Δ=0.0108)
  AUC degradation:  0.9837 → 0.9835  (Δ=0.0002)
Saved → exp4_perturbation_sensitivity.png


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_21080\969054544.py:100: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# ================================================================
# CELL 11 — Experiment 5: Calibration Impact
# ================================================================
# PRD §4 Exp 5: Compare Platt scaling vs Temperature scaling
# Measure: threshold std before vs after, EER, ECE, Brier score
# ================================================================

print("\n" + "=" * 72)
print("  EXP 5 — CALIBRATION IMPACT (Platt vs Temperature Scaling)")
print("=" * 72)

calib_rows = []
for r in fold_records:
    calib_rows.append({
        "Fold"        : r["outer_fold"],
        "ECE Raw"     : r["ece_raw"],
        "ECE Platt"   : r["ece_platt"],
        "ECE Temp"    : r["ece_temp"],
        "Temperature" : r["best_temperature"],
        "Brier Raw"   : r["youden"]["brier"],
        "Brier Platt" : r["platt"]["brier"],
        "Brier Temp"  : r["temp"]["brier"],
        "Thresh Raw"  : r["youden_thresh_mean"],
        "Thresh Platt": r["platt"]["threshold"],
        "Thresh Temp" : r["temp"]["threshold"],
        "F1 Raw"      : r["youden"]["f1"],
        "F1 Platt"    : r["platt"]["f1"],
        "F1 Temp"     : r["temp"]["f1"],
    })

df_calib = pd.DataFrame(calib_rows).set_index("Fold")
print(df_calib[["ECE Raw","ECE Platt","ECE Temp",
                "Brier Raw","Brier Platt","Brier Temp",
                "F1 Raw","F1 Platt","F1 Temp"]].to_string())

# Aggregate
print("\n  Aggregate (mean ± std):")
for col in ["ECE Raw","ECE Platt","ECE Temp",
             "Brier Raw","Brier Platt","Brier Temp",
             "F1 Raw","F1 Platt","F1 Temp"]:
    v = df_calib[col].values
    print(f"  {col:<15} {v.mean():.4f} ± {v.std():.4f}")

thresh_std_raw   = np.std([r["youden_thresh_mean"] for r in fold_records])
thresh_std_platt = np.std([r["platt"]["threshold"] for r in fold_records])
thresh_std_temp  = np.std([r["temp"]["threshold"]  for r in fold_records])
print(f"\n  Threshold std  Raw  : {thresh_std_raw:.4f}")
print(f"  Threshold std  Platt: {thresh_std_platt:.4f}")
print(f"  Threshold std  Temp : {thresh_std_temp:.4f}")

# ── Calibration plots ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: ECE comparison
ax = axes[0]
methods = ['Raw', 'Platt', 'Temp']
ece_means = [df_calib['ECE Raw'].mean(), df_calib['ECE Platt'].mean(),
             df_calib['ECE Temp'].mean()]
ece_stds  = [df_calib['ECE Raw'].std(), df_calib['ECE Platt'].std(),
             df_calib['ECE Temp'].std()]
bars = ax.bar(methods, ece_means, yerr=ece_stds,
              color=['#5c8fe0','#2ca02c','#e05c5c'],
              alpha=0.8, edgecolor='black', capsize=6)
for bar, val in zip(bars, ece_means):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
            f"{val:.4f}", ha='center', va='bottom', fontsize=10)
ax.set_ylabel("ECE (lower = better)", fontsize=10)
ax.set_title("Expected Calibration Error", fontsize=11, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.3, axis='y')

# Middle: Brier score comparison
ax = axes[1]
bs_means = [df_calib['Brier Raw'].mean(), df_calib['Brier Platt'].mean(),
            df_calib['Brier Temp'].mean()]
bs_stds  = [df_calib['Brier Raw'].std(), df_calib['Brier Platt'].std(),
            df_calib['Brier Temp'].std()]
bars = ax.bar(methods, bs_means, yerr=bs_stds,
              color=['#5c8fe0','#2ca02c','#e05c5c'],
              alpha=0.8, edgecolor='black', capsize=6)
for bar, val in zip(bars, bs_means):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
            f"{val:.4f}", ha='center', va='bottom', fontsize=10)
ax.set_ylabel("Brier Score (lower = better)", fontsize=10)
ax.set_title("Brier Score", fontsize=11, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.3, axis='y')

# Right: Threshold std before vs after
ax = axes[2]
t_stds = [thresh_std_raw, thresh_std_platt, thresh_std_temp]
bars = ax.bar(methods, t_stds,
              color=['#5c8fe0','#2ca02c','#e05c5c'],
              alpha=0.8, edgecolor='black')
for bar, val in zip(bars, t_stds):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.0005,
            f"{val:.4f}", ha='center', va='bottom', fontsize=10)
ax.set_ylabel("Threshold Std (lower = more stable)", fontsize=10)
ax.set_title("Threshold Stability\nBefore vs After Calibration",
             fontsize=11, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.3, axis='y')

plt.suptitle("Exp 5: Calibration Impact (PRD §4)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/exp5_calibration_impact.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved → exp5_calibration_impact.png")


  EXP 5 — CALIBRATION IMPACT (Platt vs Temperature Scaling)
       ECE Raw  ECE Platt  ECE Temp  Brier Raw  Brier Platt  Brier Temp    F1 Raw  F1 Platt   F1 Temp
Fold                                                                                                 
1     0.049709   0.009271  0.050274   0.061564     0.055030    0.061262  0.936989  0.930441  0.930441
2     0.048236   0.011538  0.047027   0.054111     0.047293    0.054060  0.937485  0.948683  0.948683
3     0.037847   0.021749  0.038135   0.068694     0.065498    0.067785  0.923077  0.919722  0.919722
4     0.077206   0.013302  0.077687   0.076706     0.061968    0.076618  0.909392  0.932612  0.932612
5     0.046907   0.008929  0.044013   0.060661     0.054178    0.059730  0.934124  0.940124  0.940124

  Aggregate (mean ± std):
  ECE Raw         0.0520 ± 0.0133
  ECE Platt       0.0130 ± 0.0047
  ECE Temp        0.0514 ± 0.0137
  Brier Raw       0.0643 ± 0.0077
  Brier Platt     0.0568 ± 0.0064
  Brier Temp      0.0639 ± 0

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_21080\854849011.py:106: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# ================================================================
# CELL 12 — Required Visualisation 1: ROC Curves
# ================================================================
# PRD §6 Vis 1: ROC showing strong ranking performance
# ================================================================

fig, ax = plt.subplots(figsize=(7, 6))
mean_fpr = np.linspace(0, 1, 300)
tprs_interp = []
palette = plt.cm.tab10.colors

for i, (fpr, tpr, fold_auc) in enumerate(roc_curves):
    ax.plot(fpr, tpr, lw=1.2, alpha=0.45, color=palette[i],
            label=f"Fold {i+1}  AUC={fold_auc:.4f}")
    f_interp = interp1d(fpr, tpr, kind='linear',
                        bounds_error=False, fill_value=(0,1))
    tprs_interp.append(f_interp(mean_fpr))

mean_tpr = np.mean(tprs_interp, axis=0)
std_tpr  = np.std(tprs_interp, axis=0)
mean_auc = np.mean([d[2] for d in roc_curves])
std_auc  = np.std([d[2] for d in roc_curves])

ax.plot(mean_fpr, mean_tpr, color='black', lw=2.5,
        label=f"Mean AUC = {mean_auc:.4f} ± {std_auc:.4f}")
ax.fill_between(mean_fpr, mean_tpr-std_tpr, mean_tpr+std_tpr,
                color='grey', alpha=0.18, label="± 1 std")
ax.plot([0,1],[0,1],'k--', lw=0.8, alpha=0.5, label="Random")

ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves — Nested K-Fold Outer Test Sets\n"
             "(Strong ranking performance per PRD §6)",
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.01)
ax.grid(True, linestyle='--', alpha=0.35)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/vis1_roc_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → vis1_roc_curves.png")

Saved → vis1_roc_curves.png


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_21080\664430156.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
# ================================================================
# CELL 13 — Required Visualisation 4: DET Curves
# ================================================================
# PRD §6 Vis 4: DET curves showing Youden vs EER operating points
# DET (Detection Error Tradeoff): FAR vs FRR on normal deviate scale
# ================================================================
from scipy.stats import norm

def to_normal_deviate(rate):
    """Map [0,1] rates to normal deviate scale for DET plot."""
    rate = np.clip(rate, 1e-6, 1 - 1e-6)
    return norm.ppf(rate)

fig, ax = plt.subplots(figsize=(7, 6))

for i, (fpr, fnr) in enumerate(det_curves):
    ax.plot(to_normal_deviate(fpr), to_normal_deviate(fnr),
            lw=1.2, alpha=0.45, color=palette[i],
            label=f"Fold {i+1}")

# Overlay mean Youden and EER operating points
for strat, color, marker, label in [
    ("youden", '#5c8fe0', 'o', "Youden op. point"),
    ("eer",    '#e05c5c', 's', "EER op. point"),
]:
    far_pts = [r[strat]["far"] for r in fold_records]
    frr_pts = [r[strat]["frr"] for r in fold_records]
    ax.scatter(to_normal_deviate(far_pts), to_normal_deviate(frr_pts),
               s=80, color=color, marker=marker, zorder=6,
               label=label, edgecolors='black', linewidths=0.8)

# EER diagonal
ticks = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
tick_labels = ["0.1","0.5","1","2","5","10","20","30","40","50"]
nd_ticks = [to_normal_deviate(t) for t in ticks]
ax.plot(nd_ticks, nd_ticks, 'k--', lw=1, alpha=0.4, label="EER line")
ax.set_xticks(nd_ticks); ax.set_xticklabels(tick_labels, fontsize=8)
ax.set_yticks(nd_ticks); ax.set_yticklabels(tick_labels, fontsize=8)
ax.set_xlabel("FAR (%)", fontsize=12)
ax.set_ylabel("FRR (%)", fontsize=12)
ax.set_title("DET Curves — Youden vs EER Operating Points\n"
             "(Normal deviate scale per PRD §6)",
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.3)
xlim = (nd_ticks[0], nd_ticks[-1])
ax.set_xlim(xlim); ax.set_ylim(xlim)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/vis4_det_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → vis4_det_curves.png")

Saved → vis4_det_curves.png


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_21080\1520531045.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
# ================================================================
# CELL 14 — Final Aggregated Results Table (PRD §7)
# ================================================================
# PRD §7 Output Format: per fold + aggregated mean ± std
# All metrics: AUC, EER, threshold, J-score, F1, Sens, Spec
# ================================================================

print("\n" + "=" * 72)
print("  FINAL AGGREGATED RESULTS  (PRD §7 Output Format)")
print("=" * 72)
print(f"  Framework : Nested K-Fold  "
      f"(Outer={OUTER_SPLITS}, Inner={INNER_SPLITS})")
print(f"  Thresholds: Youden's J, EER, Fixed-0.5, Platt+Youden, Temp+Youden")
print()

agg_keys = [
    ("AUC",           "youden",              "auc"),
    ("EER value",     "eer",                 "far"),   # EER ≈ FAR@EER
    ("Youden thresh", "youden_thresh_mean",  None),
    ("EER thresh",    "eer_thresh_mean",      None),
    ("J-score",       "inner_j_scores",       None),
    ("F1 (Youden)",   "youden",              "f1"),
    ("F1 (EER)",      "eer",                 "f1"),
    ("F1 (Fixed 0.5)","fixed",               "f1"),
    ("F1 (Platt)",    "platt",               "f1"),
    ("Sens (Youden)", "youden",              "sensitivity"),
    ("Spec (Youden)", "youden",              "specificity"),
    ("FAR (Youden)",  "youden",              "far"),
    ("FRR (Youden)",  "youden",              "frr"),
    ("ECE (raw)",     "ece_raw",              None),
    ("ECE (Platt)",   "ece_platt",            None),
    ("Brier (raw)",   "youden",              "brier"),
]

print(f"  {'Metric':<22} {'Mean':>9} {'Std':>9} {'Min':>9} {'Max':>9} {'95% CI':>22}")
print(f"  {'-'*80}")

for display_name, key, subkey in agg_keys:
    if subkey is None:
        if key in fold_records[0]:
            vals = np.array([r[key] for r in fold_records])
        else:
            vals = np.array([v for r in fold_records for v in r[key]])
    else:
        vals = np.array([r[key][subkey] for r in fold_records])
    ci_lo = np.percentile(vals, 2.5)
    ci_hi = np.percentile(vals, 97.5)
    print(f"  {display_name:<22} {vals.mean():>9.4f} {vals.std():>9.4f} "
          f"{vals.min():>9.4f} {vals.max():>9.4f} "
          f"  [{ci_lo:.4f}, {ci_hi:.4f}]")

print()
print("  KEY FINDING — Threshold Instability:")
y_std_all = np.std([r["youden_thresh_mean"] for r in fold_records])
e_std_all = np.std([r["eer_thresh_mean"]    for r in fold_records])
auc_std   = np.std([r["youden"]["auc"]      for r in fold_records])
print(f"    AUC std          : {auc_std:.4f}  (stable)")
print(f"    Youden thresh std: {y_std_all:.4f}  "
      f"({'>> AUC std — INSTABILITY CONFIRMED' if y_std_all > auc_std else 'comparable'})")
print(f"    EER thresh std   : {e_std_all:.4f}")
print(f"    Ratio (Y-std/AUC-std): {y_std_all/(auc_std+1e-8):.2f}x")


  FINAL AGGREGATED RESULTS  (PRD §7 Output Format)
  Framework : Nested K-Fold  (Outer=5, Inner=3)
  Thresholds: Youden's J, EER, Fixed-0.5, Platt+Youden, Temp+Youden

  Metric                      Mean       Std       Min       Max                 95% CI
  --------------------------------------------------------------------------------
  AUC                       0.9774    0.0041    0.9719    0.9837   [0.9721, 0.9833]
  EER value                 0.1030    0.0338    0.0516    0.1360   [0.0539, 0.1356]
  Youden thresh             0.4530    0.1220    0.2255    0.5591   [0.2472, 0.5587]
  EER thresh                0.4911    0.1027    0.2988    0.5830   [0.3168, 0.5814]
  J-score                   0.8194    0.0253    0.7697    0.8634   [0.7787, 0.8587]
  F1 (Youden)               0.9282    0.0108    0.9094    0.9375   [0.9108, 0.9374]
  F1 (EER)                  0.9275    0.0139    0.9023    0.9407   [0.9045, 0.9404]
  F1 (Fixed 0.5)            0.9249    0.0130    0.8996    0.9357   [0.90

In [15]:
# ================================================================
# CELL 15 — Save Best Model + Full Results
# ================================================================

SAVE_PATH = "deepfake_threshold_stability.pth"

torch.save(
    {
        "model_state_dict" : best_model_state,
        "best_outer_fold"  : best_outer_fold,
        "best_auc"         : best_global_auc,
        "outer_splits"     : OUTER_SPLITS,
        "inner_splits"     : INNER_SPLITS,
        "fold_records"     : [
            {k: v for k, v in r.items()
             if k not in ("youden","eer","fixed","platt","temp")}
            for r in fold_records
        ],
        "perturb_results"  : perturb_results,
    },
    SAVE_PATH,
)

print(f"✅ Model + results saved → {SAVE_PATH}")
print()
print("Figures saved:")
for f in [
    "exp1_threshold_variability.png",
    "exp2_auc_vs_threshold_stability.png",
    "exp3_strategy_comparison.png",
    "exp4_perturbation_sensitivity.png",
    "exp5_calibration_impact.png",
    "vis1_roc_curves.png",
    "vis4_det_curves.png",
]:
    print(f"  {f}")
print()
print("PRD §11 Definition of Done:")
print("  ✅ Nested CV with AUC, EER, thresholds")
print("  ✅ Threshold variability demonstrated (Exp 1 + 2)")
print("  ✅ Youden vs EER comparison (Exp 3)")
print("  ✅ Perturbation sensitivity (Exp 4)")
print("  ✅ Calibration impact (Exp 5)")
print("  ✅ 7 figures generated")
print("  ✅ Full narrative: instability → deployment risk")

✅ Model + results saved → deepfake_threshold_stability.pth

Figures saved:
  exp1_threshold_variability.png
  exp2_auc_vs_threshold_stability.png
  exp3_strategy_comparison.png
  exp4_perturbation_sensitivity.png
  exp5_calibration_impact.png
  vis1_roc_curves.png
  vis4_det_curves.png

PRD §11 Definition of Done:
  ✅ Nested CV with AUC, EER, thresholds
  ✅ Threshold variability demonstrated (Exp 1 + 2)
  ✅ Youden vs EER comparison (Exp 3)
  ✅ Perturbation sensitivity (Exp 4)
  ✅ Calibration impact (Exp 5)
  ✅ 7 figures generated
  ✅ Full narrative: instability → deployment risk
